In [1]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold

In [2]:

import os
import csv
import json
import torch

ROOT = "" 
TRAIN_CSV = os.path.join(ROOT, "train", "train.csv")
TEST_PRED_CSV = os.path.join(ROOT, "test", "test.csv")
SUBMISSION_CSV = "submission_dummy.csv"

def calculate_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-6)

def my_nms(boxes, scores, iou_thresh):
    I = sorted(range(len(scores)), key=lambda idx: scores[idx], reverse=True)
    K = []
    while (len(I)>0):
        i = I[0]
        K.append(i)
        I.pop(0)

        I = [j for j in I if calculate_iou(boxes[i], boxes[j]) <= iou_thresh]
        
    return K


def create_submission(test_csv_path, output_csv_path, nms_iou_threshold=0.05):
    submission_rows = []
    
    # 1. Read the provided test csv with Faster R-CNN predictions
    with open(test_csv_path, "r", encoding="utf-8") as f:
        reader = list(csv.DictReader(f))
        
    # 2. Iterate through rows. The index in this loop becomes the datapointID!
    for idx, row in enumerate(reader):
        # Parse strings into tensors

        # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        # It's important to call json.loads(...) to convert the string representation of lists 
        # back into actual lists before converting to tensors.

        boxes = torch.tensor(json.loads(row["pred_boxes"]), dtype=torch.float32) 
        scores = torch.tensor(json.loads(row["pred_scores"]), dtype=torch.float32)

        # =========================================================================================
        # UNDERSTANDING THE DETECTOR'S OUTPUT:
        # 
        # 'pred_boxes': A list of bounding boxes predicted by the Faster R-CNN model. 
        #               Each box is represented as [x_min, y_min, x_max, y_max], which are the 
        #               pixel coordinates of the top-left and bottom-right corners.
        #
        # 'pred_scores': The AI's confidence score (ranging from 0.0 to 1.0) for each box. 
        #                A score of 0.95 means the AI is 95% sure there is a person there.
        #
        # REMEMBER: Because Professor Grammar set the threshold extremely low (0.05), the model 
        #           is "over-predicting." You will see many overlapping boxes for the same person, 
        #           and many low-scoring boxes on random background objects. 
        #           Your goal is to use NMS (and later, whatever you want) to filter this list down!
        # You might want to overlay these boxes on the original images to visually understand what the model is predicting and 
        #           how NMS is helping to clean it up. SEE EXTRA DETAILS AT THE END OF THIS SCRIPT. 
        # USING THE IMAGES IS NOT REQUIRED FOR YOUR ALGORITHM!       
        # =========================================================================================
        
        # Ensure they are sorted by score descending before running NMS
        if scores.numel() > 0:
            sort_idx = scores.argsort(descending=True)
            boxes = boxes[sort_idx]
            scores = scores[sort_idx]
        
        # Apply NMS
        keep_indices = my_nms(boxes, scores, iou_thresh=nms_iou_threshold)
        
        # Get final boxes and format them back to a JSON string
        final_boxes = boxes[keep_indices].tolist()
        answer_str = json.dumps(final_boxes) # !!!!!!!!!!!!!! very important to convert back to string for CSV output!
        
        # Append for Subtask 1
        submission_rows.append([1, idx, answer_str])
        
        # Append for Subtask 2 (using the same NMS logic for both to get a valid file)
        # In a real scenario, you would run your different algorithm for subtask 2 here and 
        # append those results instead.
        submission_rows.append([2, idx, answer_str])

    # 3. Write out the final CSV
    with open(output_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["subtaskID", "datapointID", "answer"])
        writer.writerows(submission_rows)
        
    print(f"Successfully created {output_csv_path} with {len(submission_rows)} rows!")

## hide

In [3]:

import numpy as np

import numpy as np
import json
import csv

def box_iou_np(boxes1, boxes2):
    """
    Calculates Intersection over Union (IoU) between two sets of boxes.
    boxes1: numpy array of shape (N, 4) in format [x1, y1, x2, y2]
    boxes2: numpy array of shape (M, 4) in format [x1, y1, x2, y2]
    Returns: numpy array of shape (N, M) containing IoU scores.
    """
    boxes1 = np.array(boxes1, dtype=np.float32)
    boxes2 = np.array(boxes2, dtype=np.float32)

    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    area1 = np.maximum(0, boxes1[:, 2] - boxes1[:, 0]) * np.maximum(0, boxes1[:, 3] - boxes1[:, 1])
    area2 = np.maximum(0, boxes2[:, 2] - boxes2[:, 0]) * np.maximum(0, boxes2[:, 3] - boxes2[:, 1])

    lt = np.maximum(boxes1[:, None, :2], boxes2[None, :, :2])  # top-left
    rb = np.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])  # bottom-right

    wh = np.maximum(0, rb - lt)
    inter = wh[:, :, 0] * wh[:, :, 1]

    union = area1[:, None] + area2[None, :] - inter
    return inter / np.maximum(union, 1e-6)


def evaluate_detection_np(pred_boxes, gt_boxes, iou_thr=0.5):
    """
    Evaluates one image using greedy one-to-one matching.

    IMPORTANT:
    pred_boxes must be sorted by confidence score (highest to lowest).

    Returns a dictionary with:
      tp, fp, fn, precision, recall, f1, f2
    """
    pred_boxes = np.array(pred_boxes, dtype=np.float32)
    gt_boxes = np.array(gt_boxes, dtype=np.float32)

    # Handle edge cases
    if len(pred_boxes) == 0 and len(gt_boxes) == 0:
        return {
            "tp": 0, "fp": 0, "fn": 0,
            "precision": 1.0, "recall": 1.0,
            "f1": 1.0, "f2": 1.0
        }

    if len(pred_boxes) == 0:
        fn = len(gt_boxes)
        return {
            "tp": 0, "fp": 0, "fn": fn,
            "precision": 0.0, "recall": 0.0,
            "f1": 0.0, "f2": 0.0
        }

    if len(gt_boxes) == 0:
        fp = len(pred_boxes)
        return {
            "tp": 0, "fp": fp, "fn": 0,
            "precision": 0.0, "recall": 0.0,
            "f1": 0.0, "f2": 0.0
        }

    ious = box_iou_np(pred_boxes, gt_boxes)
    gt_used = np.zeros(len(gt_boxes), dtype=bool)

    tp = 0
    fp = 0

    # Greedy matching
    for p in range(len(pred_boxes)):
        row = ious[p].copy()
        row[gt_used] = -1.0

        best_g = np.argmax(row)
        best_iou = row[best_g]

        if best_iou >= iou_thr:
            tp += 1
            gt_used[best_g] = True
        else:
            fp += 1

    fn = np.sum(~gt_used)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision == 0 and recall == 0:
        f1 = 0.0
        f2 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
        f2 = (1 + 2**2) * precision * recall / (2**2 * precision + recall)

    return {
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "f2": float(f2),
    }


def evaluate_train_baseline(train_csv_path):
    print(f"Loading training data from: {train_csv_path}")
    with open(train_csv_path, "r", encoding="utf-8") as f:
        train_rows = list(csv.DictReader(f))

    total_tp = 0
    total_fp = 0
    total_fn = 0

    f1_scores = []
    f2_scores = []

    print("Evaluating baseline (Score-Only) performance on training set...")

    for row in train_rows:
        pred_boxes_list = json.loads(row["pred_boxes"])
        pred_scores_list = json.loads(row["pred_scores"])
        gt_boxes_list = json.loads(row["gt_boxes"])

        # Sort predicted boxes by score descending
        if len(pred_scores_list) > 0:
            paired = list(zip(pred_scores_list, pred_boxes_list))
            paired.sort(key=lambda x: x[0], reverse=True)
            pred_boxes_list = [box for score, box in paired]

        iou_thr=0.5
        result = evaluate_detection_np(pred_boxes_list, gt_boxes_list, iou_thr)

        total_tp += result["tp"]
        total_fp += result["fp"]
        total_fn += result["fn"]

        f1_scores.append(result["f1"])
        f2_scores.append(result["f2"])

    # Global precision / recall / F1 / F2
    global_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    global_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0

    if global_precision == 0 and global_recall == 0:
        global_f1 = 0.0
        global_f2 = 0.0
    else:
        global_f1 = 2 * global_precision * global_recall / (global_precision + global_recall)
        global_f2 = (1 + 2**2) * global_precision * global_recall / (2**2 * global_precision + global_recall)

    # !!!! Here is the official score that will be used for ranking in the competition
    official_score = 0.4 * global_f2 + 0.6 * global_f1

    print("-" * 50)
    print("TRAIN SET BASELINE RESULTS (Score threshold = 0.05)")
    print("-" * 50)
    print(f"Total True Positives (TP):   {total_tp}")
    print(f"Total False Positives (FP):  {total_fp}")
    print(f"Total False Negatives (FN):  {total_fn}")
    print(f"Global Precision:            {global_precision:.4f}")
    print(f"Global Recall:               {global_recall:.4f}")
    print(f"Global F1@0.5:               {global_f1:.4f}")
    print(f"Global F2@0.5:               {global_f2:.4f}")
    print(f"Mean image-level F1@0.5:     {np.mean(f1_scores):.4f}")
    print(f"Mean image-level F2@0.5:     {np.mean(f2_scores):.4f}")
    print("-" * 50)
    print("If your NMS / filtering algorithm is working, FP should usually go down.")

    print("-" * 50)
    print(f"Official score (for leaderboard):  {official_score:.4f}")
    print("-" * 50)

## . 

In [4]:

def evaluate_nms(train_csv_path, tau):
   # print(f"Loading training data from: {train_csv_path}")
    with open(train_csv_path, "r", encoding="utf-8") as f:
        train_rows = list(csv.DictReader(f))

    total_tp = 0
    total_fp = 0
    total_fn = 0

    f1_scores = []
    f2_scores = []

    for row in train_rows:
        pred_boxes_list = json.loads(row["pred_boxes"])
        pred_scores_list = json.loads(row["pred_scores"])
        gt_boxes_list = json.loads(row["gt_boxes"])

        # Sort predicted boxes by score descending
        if len(pred_scores_list) > 0:
            paired = list(zip(pred_scores_list, pred_boxes_list))
            paired.sort(key=lambda x: x[0], reverse=True)
            pred_boxes_list = [box for score, box in paired]
            pred_scores_list = [score for score, box in paired]

        keep = my_nms(pred_boxes_list, pred_scores_list, iou_thresh=tau)
        keptbox = [pred_boxes_list[i] for i in keep]
        result = evaluate_detection_np(keptbox, gt_boxes_list, iou_thr=0.5)
        

        total_tp += result["tp"]
        total_fp += result["fp"]
        total_fn += result["fn"]

        f1_scores.append(result["f1"])
        f2_scores.append(result["f2"])

    # Global precision / recall / F1 / F2
    global_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    global_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0

    if global_precision == 0 and global_recall == 0:
        global_f1 = 0.0
        global_f2 = 0.0
    else:
        global_f1 = 2 * global_precision * global_recall / (global_precision + global_recall)
        global_f2 = (1 + 2**2) * global_precision * global_recall / (2**2 * global_precision + global_recall)

    # !!!! Here is the official score that will be used for ranking in the competition
    official_score = 0.4 * global_f2 + 0.6 * global_f1

    '''
    print("-" * 50)
    print("TRAIN SET BASELINE RESULTS (Score threshold = 0.05)")
    print("-" * 50)
    print(f"Total True Positives (TP):   {total_tp}")
    print(f"Total False Positives (FP):  {total_fp}")
    print(f"Total False Negatives (FN):  {total_fn}")
    print(f"Global Precision:            {global_precision:.4f}")
    print(f"Global Recall:               {global_recall:.4f}")
    print(f"Global F1@0.5:               {global_f1:.4f}")
    print(f"Global F2@0.5:               {global_f2:.4f}")
    print(f"Mean image-level F1@0.5:     {np.mean(f1_scores):.4f}")
    print(f"Mean image-level F2@0.5:     {np.mean(f2_scores):.4f}")
    print("-" * 50)
    print("If your NMS / filtering algorithm is working, FP should usually go down.")

    print("-" * 50)
    print(f"Official score (for leaderboard):  {official_score:.4f}")
    print("-" * 50)
    '''

    return official_score


In [5]:

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def visualize_boxes(image_path, boxes, scores=None, color='red', title="Bounding Boxes"):
    """
    Reads an image and overlays a list of bounding boxes.
    
    Parameters:
    - image_path (str): Path to the image file.
    - boxes (list or array): List of bounding boxes in [x_min, y_min, x_max, y_max] format.
    - scores (list or array, optional): Confidence scores to display above each box.
    - color (str, optional): Color of the bounding boxes (e.g., 'red', 'green', 'blue').
    - title (str, optional): Title for the plot.
    """
    try:
        # Open the image
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return
    
    # Create figure and axes
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)
    ax.set_title(title)
    
    # Loop over the boxes and add them to the plot
    for i, box in enumerate(boxes):
        x_min, y_min, x_max, y_max = box
        width = x_max - x_min
        height = y_max - y_min
        
        # Create a Rectangle patch
        rect = patches.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # If scores are provided, add them as text slightly above the box
        if scores is not None and i < len(scores):
            score_text = f"{scores[i]:.2f}"
            ax.text(x_min, y_min - 5, score_text, color='white', fontsize=10, fontweight='bold',
                    bbox=dict(facecolor=color, alpha=0.7, pad=2, edgecolor='none'))
            
    # Remove axes ticks for a cleaner look
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [6]:
for tau in [0.05, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    print(tau, evaluate_nms(TRAIN_CSV, tau))

BEST_TAU = 0.05

0.05 0.6689244765578788
0.2 0.62417150020184
0.3 0.604145656295207
0.4 0.5616275521232903
0.5 0.4486174744795435
0.6 0.4486174744795435
0.7 0.4486174744795435


# submission

In [7]:
 create_submission(TEST_PRED_CSV, 'submission.csv', nms_iou_threshold=BEST_TAU)

Successfully created submission.csv with 136 rows!


# subtask2

In [8]:
CAND_TAU = 0.3

In [9]:
def box_features(boxes, scores, tau):
    keep = my_nms(boxes, scores, iou_thresh=tau)
    rows = []
    for rank in range(len(keep)):
        i = keep[rank]
        b = boxes[i]
        w = b[2] - b[0]
        h = b[3] - b[1]
        support = 0                      
        support_score = 0.0
        for j in range(len(boxes)):
            if j != i and calculate_iou(b, boxes[j]) > 0.5:
                support += 1
                support_score += scores[j]
        max_iou_better = 0
        contained = 0
        for r2 in range(rank):
            other = boxes[keep[r2]]
            max_iou_better = max(max_iou_better, calculate_iou(b, other))
            ix = max(0, min(b[2], other[2]) - max(b[0], other[0]))
            iy = max(0, min(b[3], other[3]) - max(b[1], other[1]))
            contained = max(contained, ix * iy / (w * h + 1e-6))
        rows.append({
            'score': scores[i],
            'rank': rank,                        
            'rel_score': scores[i] / scores[0], 
            'n_kept': len(keep),
            'w': w, 'h': h, 'area': w * h,
            'ratio': h / (w + 1e-6),            
            'x_center': (b[0] + b[2]) / 2,
            'bottom': b[3],                      
            'support': support,
            'support_score': support_score,
            'max_iou_better': max_iou_better,
            'contained': contained
        })
    return keep, rows

In [10]:
table = []
labels = []
groups = []     
kept_boxes = []  

with open(TRAIN_CSV, "r", encoding="utf-8") as f:
    train_rows = list(csv.DictReader(f))

for img_id in range(len(train_rows)):
    row = train_rows[img_id]
    boxes = json.loads(row["pred_boxes"])
    scores = json.loads(row["pred_scores"])
    gt = json.loads(row["gt_boxes"])

    order = sorted(range(len(scores)), key=lambda k: scores[k], reverse=True)
    boxes = [boxes[k] for k in order]
    scores = [scores[k] for k in order]

    keep, rows = box_features(boxes, scores, CAND_TAU)
    gt_used = [False] * len(gt)
    for n in range(len(keep)):                 
        b = boxes[keep[n]]
        best_g = -1
        best_iou = 0.5
        for g in range(len(gt)):
            if not gt_used[g]:
                iou = calculate_iou(b, gt[g])
                if iou >= best_iou:
                    best_iou = iou
                    best_g = g
        if best_g >= 0:
            gt_used[best_g] = True              
            labels.append(1)
        else:
            labels.append(0)                   
        table.append(rows[n])
        groups.append(img_id)
        kept_boxes.append(b)

X = pd.DataFrame(table)
y = np.array(labels)
groups = np.array(groups)
print(X.shape, y.mean())    

(690, 14) 0.3565217391304348


In [11]:
oof = np.zeros(len(y))
for fit_rows, val_rows in GroupKFold(n_splits=5).split(X, y, groups):
    model = CatBoostClassifier(iterations=300, verbose=0, random_seed=0)
    model.fit(X.iloc[fit_rows], y[fit_rows])
    oof[val_rows] = model.predict_proba(X.iloc[val_rows])[:, 1]

In [12]:
def official(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp > 0 else 0
    r = tp / (tp + fn) if tp + fn > 0 else 0
    if p + r == 0:
        return 0
    f1 = 2 * p * r / (p + r)
    f2 = 5 * p * r / (4 * p + r)
    return 0.4 * f2 + 0.6 * f1

for t in [0.2, 0.3, 0.4, 0.5, 0.6]:
    tp = fp = fn = 0
    for img_id in range(len(train_rows)):
        chosen = []
        for k in np.where(groups == img_id)[0]:
            if oof[k] >= t:
                chosen.append(k)
        chosen.sort(key=lambda k: oof[k], reverse=True)    # the evaluator expects best first
        res = evaluate_detection_np([kept_boxes[k] for k in chosen], json.loads(train_rows[img_id]["gt_boxes"]))
        tp += res["tp"]
        fp += res["fp"]
        fn += res["fn"]
    print(t, round(official(tp, fp, fn), 4), 'tp', tp, 'fp', fp, 'fn', fn)


0.2 0.9196 tp 235 fp 28 fn 16
0.3 0.9174 tp 232 fp 24 fn 19
0.4 0.9111 tp 228 fp 21 fn 23
0.5 0.9074 tp 225 fp 18 fn 26
0.6 0.9104 tp 224 fp 14 fn 27


In [15]:
BEST_CUTOFF = 0.2

final_model = CatBoostClassifier(iterations=300, verbose=0, random_seed=0)
final_model.fit(X, y)                                   # all training boxes this time

with open(TEST_PRED_CSV, "r", encoding="utf-8") as f:
    test_rows = list(csv.DictReader(f))

submission_rows = []
for idx in range(len(test_rows)):
    row = test_rows[idx]
    boxes = json.loads(row["pred_boxes"])
    scores = json.loads(row["pred_scores"])

    order = sorted(range(len(scores)), key=lambda k: scores[k], reverse=True)
    boxes = [boxes[k] for k in order]
    scores = [scores[k] for k in order]

    keep, rows = box_features(boxes, scores, 0.1)

    # subtask 1: plain NMS
    nms_boxes = [boxes[i] for i in keep]
    submission_rows.append([1, idx, json.dumps(nms_boxes)])

    keep, rows = box_features(boxes, scores, CAND_TAU)

    # subtask 2: NMS, then keep only boxes the classifier trusts
    chosen = []
    if len(rows) > 0:
        probs = final_model.predict_proba(pd.DataFrame(rows))[:, 1]
        for n in range(len(keep)):
            if probs[n] >= BEST_CUTOFF:
                chosen.append((probs[n], boxes[keep[n]]))
        chosen.sort(key=lambda p: p[0], reverse=True)       # most likely first
    submission_rows.append([2, idx, json.dumps([b for p, b in chosen])])

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["subtaskID", "datapointID", "answer"])
    writer.writerows(submission_rows)
print(len(submission_rows))

136
